<a href="https://colab.research.google.com/github/rufinapricy/fastai-notebooks/blob/main/nbs/03_backprop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The forward and backward passes

In [7]:
from torchvision.datasets import MNIST
from pathlib import Path
import gzip, pickle

path_data = Path("data")
path_data.mkdir(exist_ok=True)

print("Downloading MNIST dataset...")

train_ds = MNIST("data", train=True, download=True)
test_ds = MNIST("data", train=False, download=True)

# Convert images to normalized tensors
x_train = train_ds.data.float().view(-1, 784) / 255
y_train = train_ds.targets

x_valid = x_train[50000:]
y_valid = y_train[50000:]

x_train = x_train[:50000]
y_train = y_train[:50000]

x_test = test_ds.data.float().view(-1, 784) / 255
y_test = test_ds.targets

# Convert to the format Jeremy's notebook expects
dataset = (
    (x_train.numpy(), y_train.numpy()),
    (x_valid.numpy(), y_valid.numpy()),
    (x_test.numpy(), y_test.numpy())
)

path_gz = path_data / "mnist.pkl.gz"

with gzip.open(path_gz, "wb") as f:
    pickle.dump(dataset, f)

print("Dataset ready!")
print("Training:", x_train.shape)
print("Validation:", x_valid.shape)

100%|██████████| 9.91M/9.91M [00:00<00:00, 61.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.49MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.2MB/s]


Dataset ready!
Training: torch.Size([50000, 784])
Validation: torch.Size([10000, 784])


In [8]:
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl, numpy as np
from pathlib import Path
from torch import tensor
from fastcore.test import test_close
torch.manual_seed(42)

mpl.rcParams['image.cmap'] = 'gray'
torch.set_printoptions(precision=2, linewidth=125, sci_mode=False)
np.set_printoptions(precision=2, linewidth=125)

path_data = Path('data')
path_gz = path_data/'mnist.pkl.gz'
with gzip.open(path_gz, 'rb') as f: ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')
x_train, y_train, x_valid, y_valid = map(tensor, [x_train, y_train, x_valid, y_valid])

## Foundations version

### Basic architecture

In [9]:
n,m = x_train.shape
c = y_train.max()+1
n,m,c

(50000, 784, tensor(10))

In [10]:
# num hidden
nh = 50

In [11]:
w1 = torch.randn(m,nh)
b1 = torch.zeros(nh)
w2 = torch.randn(nh,1)
b2 = torch.zeros(1)

In [12]:
def lin(x, w, b): return x@w + b

In [13]:
t = lin(x_valid, w1, b1)
t.shape

torch.Size([10000, 50])

In [15]:
def relu(x): return x.clamp_min(0.)

In [16]:
t = relu(t)
t

tensor([[ 0.00, 11.92,  0.00,  ...,  5.50,  2.15, 15.36],
        [ 5.40, 10.25,  0.00,  ...,  0.88,  0.08, 20.31],
        [ 3.33,  0.12,  3.11,  ..., 16.96,  0.00, 24.84],
        ...,
        [ 4.02, 10.39,  0.00,  ...,  0.23,  0.00, 18.35],
        [10.66,  0.00, 10.76,  ...,  0.00,  0.00, 18.30],
        [ 2.85,  0.00,  1.43,  ...,  0.00,  5.78,  2.13]])

In [17]:
def model(xb):
    l1 = lin(xb, w1, b1)
    l2 = relu(l1)
    return lin(l2, w2, b2)

In [18]:
res = model(x_valid)
res.shape

torch.Size([10000, 1])

### Loss function: MSE

(Of course, `mse` is not a suitable loss function for multi-class classification; we'll use a better loss function soon. We'll use `mse` for now to keep things simple.)

In [19]:
res.shape,y_valid.shape

(torch.Size([10000, 1]), torch.Size([10000]))

In [20]:
(res-y_valid).shape

torch.Size([10000, 10000])

We need to get rid of that trailing (,1), in order to use `mse`.

In [21]:
res[:,0].shape

torch.Size([10000])

In [22]:
res.squeeze().shape

torch.Size([10000])

In [23]:
(res[:,0]-y_valid).shape

torch.Size([10000])

In [24]:
y_train,y_valid = y_train.float(),y_valid.float()

preds = model(x_train)
preds.shape

torch.Size([50000, 1])

In [25]:
def mse(output, targ): return (output[:,0]-targ).pow(2).mean()

In [26]:
mse(preds, y_train)

tensor(4340.72)

### Gradients and backward pass

In [27]:
from sympy import symbols,diff
x,y = symbols('x y')
diff(x**2, x)

2*x

In [28]:
diff(3*x**2+9, x)

6*x

In [29]:
def lin_grad(inp, out, w, b):
    # grad of matmul with respect to input
    inp.g = out.g @ w.t()
    w.g = (inp.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)
    b.g = out.g.sum(0)

In [30]:
def forward_and_backward(inp, targ):
    # forward pass:
    l1 = lin(inp, w1, b1)
    l2 = relu(l1)
    out = lin(l2, w2, b2)
    diff = out[:,0]-targ
    loss = diff.pow(2).mean()

    # backward pass:
    out.g = 2.*diff[:,None] / inp.shape[0]
    lin_grad(l2, out, w2, b2)
    l1.g = (l1>0).float() * l2.g
    lin_grad(inp, l1, w1, b1)

In [31]:
forward_and_backward(x_train, y_train)

In [32]:
# Save for testing against later
def get_grad(x): return x.g.clone()
chks = w1,w2,b1,b2,x_train
grads = w1g,w2g,b1g,b2g,ig = tuple(map(get_grad, chks))

We cheat a little bit and use PyTorch autograd to check our results.

In [33]:
def mkgrad(x): return x.clone().requires_grad_(True)
ptgrads = w12,w22,b12,b22,xt2 = tuple(map(mkgrad, chks))

In [34]:
def forward(inp, targ):
    l1 = lin(inp, w12, b12)
    l2 = relu(l1)
    out = lin(l2, w22, b22)
    return mse(out, targ)

In [35]:
loss = forward(xt2, y_train)
loss.backward()

In [36]:
for a,b in zip(grads, ptgrads): test_close(a, b.grad, eps=0.01)

## Refactor model

### Layers as classes

In [37]:
class Relu():
    def __call__(self, inp):
        self.inp = inp
        self.out = inp.clamp_min(0.)
        return self.out

    def backward(self): self.inp.g = (self.inp>0).float() * self.out.g

In [38]:
class Lin():
    def __init__(self, w, b): self.w,self.b = w,b

    def __call__(self, inp):
        self.inp = inp
        self.out = lin(inp, self.w, self.b)
        return self.out

    def backward(self):
        self.inp.g = self.out.g @ self.w.t()
        self.w.g = self.inp.t() @ self.out.g
        self.b.g = self.out.g.sum(0)

In [39]:
class Mse():
    def __call__(self, inp, targ):
        self.inp,self.targ = inp,targ
        self.out = mse(inp, targ)
        return self.out

    def backward(self):
        self.inp.g = 2. * (self.inp.squeeze() - self.targ).unsqueeze(-1) / self.targ.shape[0]

In [40]:
class Model():
    def __init__(self, w1, b1, w2, b2):
        self.layers = [Lin(w1,b1), Relu(), Lin(w2,b2)]
        self.loss = Mse()

    def __call__(self, x, targ):
        for l in self.layers: x = l(x)
        return self.loss(x, targ)

    def backward(self):
        self.loss.backward()
        for l in reversed(self.layers): l.backward()

In [41]:
model = Model(w1, b1, w2, b2)

In [42]:
loss = model(x_train, y_train)

In [43]:
model.backward()

In [44]:
test_close(w2g, w2.g, eps=0.01)
test_close(b2g, b2.g, eps=0.01)
test_close(w1g, w1.g, eps=0.01)
test_close(b1g, b1.g, eps=0.01)
test_close(ig, x_train.g, eps=0.01)

### Module.forward()

In [45]:
class Module():
    def __call__(self, *args):
        self.args = args
        self.out = self.forward(*args)
        return self.out

    def forward(self): raise Exception('not implemented')
    def backward(self): self.bwd(self.out, *self.args)
    def bwd(self): raise Exception('not implemented')

In [46]:
class Relu(Module):
    def forward(self, inp): return inp.clamp_min(0.)
    def bwd(self, out, inp): inp.g = (inp>0).float() * out.g

In [47]:
class Lin(Module):
    def __init__(self, w, b): self.w,self.b = w,b
    def forward(self, inp): return inp@self.w + self.b
    def bwd(self, out, inp):
        inp.g = self.out.g @ self.w.t()
        self.w.g = inp.t() @ self.out.g
        self.b.g = self.out.g.sum(0)

In [48]:
class Mse(Module):
    def forward (self, inp, targ): return (inp.squeeze() - targ).pow(2).mean()
    def bwd(self, out, inp, targ): inp.g = 2*(inp.squeeze()-targ).unsqueeze(-1) / targ.shape[0]

In [49]:
model = Model(w1, b1, w2, b2)

In [50]:
loss = model(x_train, y_train)

In [51]:
model.backward()

In [52]:
test_close(w2g, w2.g, eps=0.01)
test_close(b2g, b2.g, eps=0.01)
test_close(w1g, w1.g, eps=0.01)
test_close(b1g, b1.g, eps=0.01)
test_close(ig, x_train.g, eps=0.01)

### Autograd

In [53]:
from torch import nn
import torch.nn.functional as F

In [54]:
class Linear(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.w = torch.randn(n_in,n_out).requires_grad_()
        self.b = torch.zeros(n_out).requires_grad_()
    def forward(self, inp): return inp@self.w + self.b

In [55]:
class Model(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.layers = [Linear(n_in,nh), nn.ReLU(), Linear(nh,n_out)]

    def __call__(self, x, targ):
        for l in self.layers: x = l(x)
        return F.mse_loss(x, targ[:,None])

In [56]:
model = Model(m, nh, 1)
loss = model(x_train, y_train)
loss.backward()

In [57]:
l0 = model.layers[0]
l0.b.grad

tensor([-19.66,  -2.41,  -0.12,   2.00,  12.82, -15.36, -18.49,   0.35,   3.76,  14.71,  10.85,  12.24,  -2.96, -28.40,
          0.76,  69.37, -21.93,  49.94,  -7.10,   1.45,  25.28,  11.30, -18.21, -13.17, -17.74, -10.45,  -0.13, -18.95,
        -34.90,  -0.84,  41.01,   4.46,  62.54,  31.80,  55.32,  45.27,   3.26,  12.79,  12.48,  -1.42,   4.56,  -6.04,
        -62.70,  -1.89,  -1.41,   7.02,   0.49,  18.78,  -4.85,  -6.54])